# Baseline Comparison: PRIME vs RAOM4CC vs EdgeWiseCR

Statistical comparison of the pricing-driven resource allocation optimiser (PRIME) against
15 heuristic baselines derived from two papers: RAOM4CC (9 variants) and EdgeWiseCR (6 variants).

**Timing note:** PRIME execution times use server-side solver time (`completed_at - started_at`),
excluding HTTP overhead (~1.4% of wall clock).

**Cost note:** Cost comparison is excluded because baselines model only 5/12 constraint types.
Solutions from baselines may violate provider exclusions, feature requirements, and other
constraints that PRIME enforces. Therefore, cost figures from baselines are not directly
comparable to PRIME's.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from datetime import datetime
import json, os, ast, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)
RESULTS_DIR = 'results'
TOPO_DIR = 'synthetic-dataset/synthetic-topologies'

FAMILY_COLORS = {'PRIME': '#3498db', 'RAOM4CC': '#e74c3c', 'EdgeWiseCR': '#2ecc71'}
def tech_color(tech):
    for fam, c in FAMILY_COLORS.items():
        if tech.startswith(fam): return c
    return '#95a5a6'

# Constraint coverage per technique family
# Hard constraints enforced as placement rules
HARD_CONSTRAINTS = {'PRIME': 12, 'RAOM4CC': 6, 'EdgeWiseCR': 6}
# Diagnostic metrics computed but not enforced as hard constraints
DIAGNOSTIC_METRICS = {'PRIME': 0, 'RAOM4CC': 2, 'EdgeWiseCR': 0}
# Total constraint awareness (hard + diagnostic)
CONSTRAINT_COVERAGE = {k: HARD_CONSTRAINTS[k] + DIAGNOSTIC_METRICS[k]
                       for k in HARD_CONSTRAINTS}

## 1. Data Loading and Cleaning

In [ ]:
# --- Load raw CSVs ---
prime_raw = pd.read_csv(f'{RESULTS_DIR}/results.csv')
raom_raw  = pd.read_csv(f'{RESULTS_DIR}/raom4cc_benchmark_results.csv')
ew_raw    = pd.read_csv(f'{RESULTS_DIR}/edgewisecr_results.csv')

# --- Parse PRIME (server-side solver time) ---
def compute_solver_time(row):
    try:
        sta = datetime.fromisoformat(row['started_at'].replace('Z', '+00:00'))
        com = datetime.fromisoformat(row['completed_at'].replace('Z', '+00:00'))
        return (com - sta).total_seconds()
    except:
        return float(row['time_seconds'])

prime = pd.DataFrame({
    'scenario_id': prime_raw['scenario_id'], 'technique': 'PRIME',
    'time_s': prime_raw.apply(compute_solver_time, axis=1),
    'feasible': (prime_raw['status'] == 'COMPLETED'),
    'nodes': prime_raw['add_ons'].apply(lambda x: len(ast.literal_eval(x)) if pd.notna(x) else 0),
    'family': 'PRIME',
})

raom = pd.DataFrame({
    'scenario_id': raom_raw['scenario_id'], 'technique': 'RAOM4CC_' + raom_raw['algorithm'],
    'time_s': raom_raw['time_seconds'].astype(float),
    'feasible': (raom_raw['feasible'].astype(str) == 'True'),
    'nodes': raom_raw['selected_node'].apply(lambda x: len(ast.literal_eval(x)) if pd.notna(x) else 0),
    'family': 'RAOM4CC',
})

ew = pd.DataFrame({
    'scenario_id': ew_raw['scenario_id'], 'technique': 'EdgeWiseCR_' + ew_raw['algorithm'],
    'time_s': ew_raw['time_seconds'].astype(float),
    'feasible': ew_raw['feasible'].astype(str) == 'True',
    'nodes': ew_raw['bins'].astype(int), 'family': 'EdgeWiseCR',
})

all_data = pd.concat([prime, raom, ew], ignore_index=True)

# --- Extract scenario metadata ---
meta = all_data['scenario_id'].apply(lambda sid: pd.Series(dict(zip(
    ['scale', 'vary', 'app'], sid.split('_')[:3]))))
all_data = pd.concat([all_data, meta], axis=1)

# --- Load device counts from topology metadata ---
topo_to_devices = {}
for tid in os.listdir(TOPO_DIR):
    meta_path = os.path.join(TOPO_DIR, tid, 'metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            m = json.load(f)
        topo_to_devices[tid] = m['num_devices']

# Map scenario_id -> topology_id -> num_devices
raom_topo = dict(zip(raom_raw['scenario_id'], raom_raw['topology_id']))
ew_topo = dict(zip(ew_raw['scenario_id'], ew_raw['topology_id']))
prime_topo = {}
# PRIME doesn't have topology_id directly; use raom mapping
for sid in prime['scenario_id']:
    prime_topo[sid] = raom_topo.get(sid, ew_topo.get(sid, ''))

all_data['topology_id'] = all_data['scenario_id'].map(
    lambda sid: raom_topo.get(sid, ew_topo.get(sid, '')))
all_data['num_devices'] = all_data['topology_id'].map(topo_to_devices).fillna(0).astype(int)

# --- Constraint coverage ---
all_data['constraints'] = all_data['family'].map(CONSTRAINT_COVERAGE)

feasible_data = all_data[all_data['feasible']]
feasible_all = feasible_data.copy()

print(f'Total rows: {len(all_data):,}')
print(f'Techniques: {all_data["technique"].nunique()}')
print(f'Scenarios: {all_data["scenario_id"].nunique()}')
print(f'Device count range: {all_data["num_devices"].min()} - {all_data["num_devices"].max()}')

## 2. Summary Statistics Table

In [ ]:
summary = all_data.groupby('technique').agg(
    feasible_pct=('feasible', lambda x: 100 * x.mean()),
    time_median=('time_s', 'median'),
    nodes_mean=('nodes', 'mean'),
    family=('family', 'first'),
    constraints=('constraints', 'first'),
).sort_values('family')

summary[['family', 'feasible_pct', 'time_median', 'nodes_mean', 'constraints']]

## 3. Feasibility Analysis

In [ ]:
feas = all_data.groupby('technique')['feasible'].mean().sort_values() * 100
colors = [tech_color(t) for t in feas.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(feas)), feas.values, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(feas))); ax.set_yticklabels(feas.index, fontsize=9)
ax.set_xlabel('Feasibility rate (%)'); ax.set_title('Feasibility Rate by Technique')
ax.axvline(100, color='grey', ls='--', alpha=0.4); ax.set_xlim(0, 110)
for i, v in enumerate(feas.values):
    ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9, fontweight='bold')
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc='lower right', title='Family')
plt.tight_layout()
plt.show()

## 4. Execution Time Analysis

In [ ]:
# 4a. Dot plot with IQR
time_stats = feasible_data.groupby('technique').agg(
    median=('time_s', 'median'),
    q25=('time_s', lambda x: x.quantile(0.25)),
    q75=('time_s', lambda x: x.quantile(0.75)),
    family=('family', 'first'),
).sort_values('median')

fig, ax = plt.subplots(figsize=(12, 7))
colors = [tech_color(t) for t in time_stats.index]
for i, (tech, row) in enumerate(time_stats.iterrows()):
    ax.plot([row['q25'], row['q75']], [i, i], color=colors[i], linewidth=3, alpha=0.4)
ax.scatter(time_stats['median'], range(len(time_stats)), c=colors, s=100, zorder=5, edgecolors='black', linewidth=0.5)
ax.set_yticks(range(len(time_stats))); ax.set_yticklabels(time_stats.index, fontsize=9)
ax.set_xscale('log'); ax.set_xlabel('Solver time (seconds, log scale)')
ax.set_title('Execution Time by Technique (median + IQR, feasible solutions)')
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc='lower right', title='Family')
plt.tight_layout()
plt.show()

In [ ]:
# 4b. Scatter: num_devices vs execution time (key techniques)
key_techs = ['PRIME', 'RAOM4CC_delay_heuristics', 'RAOM4CC_best_fit', 'RAOM4CC_one_layer_edge',
             'EdgeWiseCR_edgewise', 'EdgeWiseCR_prolog']
scatter_data = feasible_all[feasible_all['technique'].isin(key_techs)].copy()

# Aggregate by (technique, num_devices) for cleaner plot
scatter_agg = scatter_data.groupby(['technique', 'num_devices']).agg(
    median_time=('time_s', 'median'),
    q25=('time_s', lambda x: x.quantile(0.25)),
    q75=('time_s', lambda x: x.quantile(0.75)),
).reset_index()

fig, ax = plt.subplots(figsize=(12, 7))
for tech in key_techs:
    subset = scatter_agg[scatter_agg['technique'] == tech].sort_values('num_devices')
    c = tech_color(tech)
    ax.fill_between(subset['num_devices'], subset['q25'], subset['q75'],
                    alpha=0.15, color=c)
    ax.plot(subset['num_devices'], subset['median_time'],
            marker='o', markersize=5, linewidth=2, color=c,
            label=tech.replace('RAOM4CC_', 'RAOM. ').replace('EdgeWiseCR_', 'EW. '))

ax.set_xlabel('Number of devices in topology')
ax.set_ylabel('Median solver time (seconds)')
ax.set_yscale('log')
ax.set_title('Execution Time vs Topology Size (median + IQR)')
ax.legend(title='Technique', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 4c. Statistical tests
groups = [g['time_s'].values for _, g in feasible_data.groupby('technique')]
kw_stat, kw_p = stats.kruskal(*groups)
print(f'Kruskal-Wallis H={kw_stat:.2f}, p={kw_p:.2e}')

prime_times = feasible_data[feasible_data['technique'] == 'PRIME']['time_s'].values
print(f'\nPairwise Mann-Whitney U (PRIME vs each baseline):')
print(f'{"Baseline":<35s} {"U stat":>10s} {"p-value":>12s} {"Effect size (r)":>15s}')
print('-' * 75)
for tech in sorted(feasible_data['technique'].unique()):
    if tech == 'PRIME': continue
    tech_times = feasible_data[feasible_data['technique'] == tech]['time_s'].values
    u_stat, p_val = stats.mannwhitneyu(prime_times, tech_times, alternative='two-sided')
    n1, n2 = len(prime_times), len(tech_times)
    z = stats.norm.ppf(1 - p_val / 2)
    r = abs(z) / np.sqrt(n1 + n2)
    print(f'{tech:<35s} {u_stat:>10.0f} {p_val:>12.2e} {r:>15.3f}')

In [ ]:
# 4d. Time by scenario scale (grouped bars)
scale_order = ['small', 'medium', 'large']
scale_time = feasible_all.groupby(['scale', 'technique'])['time_s'].median().unstack(level=0)
scale_time = scale_time.reindex(columns=scale_order)
scale_time_plot = scale_time.loc[[t for t in key_techs if t in scale_time.index]]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(scale_time_plot)); width = 0.25
for i, scale in enumerate(scale_order):
    ax.bar(x + i*width, scale_time_plot[scale].values, width, label=scale, alpha=0.85, edgecolor='white')
ax.set_xticks(x + width)
ax.set_xticklabels([t.replace('RAOM4CC_', 'RAOM. ').replace('EdgeWiseCR_', 'EW.')
                     for t in scale_time_plot.index], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Median solver time (s)'); ax.set_title('Execution Time by Technique and Scenario Scale')
ax.set_yscale('log'); ax.legend(title='Scale')
plt.tight_layout()
plt.show()

## 5. Node Selection Analysis

In [ ]:
node_stats = feasible_all.groupby('technique').agg(
    median=('nodes', 'median'),
    q25=('nodes', lambda x: x.quantile(0.25)),
    q75=('nodes', lambda x: x.quantile(0.75)),
    family=('family', 'first'),
).sort_values('median')

fig, ax = plt.subplots(figsize=(12, 7))
colors = [tech_color(t) for t in node_stats.index]
for i, (tech, row) in enumerate(node_stats.iterrows()):
    ax.plot([row['q25'], row['q75']], [i, i], color=colors[i], linewidth=3, alpha=0.4)
ax.scatter(node_stats['median'], range(len(node_stats)), c=colors, s=100, zorder=5, edgecolors='black', linewidth=0.5)
ax.set_yticks(range(len(node_stats))); ax.set_yticklabels(node_stats.index, fontsize=9)
ax.set_xlabel('Number of selected nodes')
ax.set_title('Node Count by Technique (median + IQR, feasible solutions)')
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc='lower right', title='Family')
plt.tight_layout()
plt.show()

## 6. Pareto Front: Execution Time vs Constraint Coverage

Each technique is evaluated on two dimensions:
- **X-axis:** Median execution time (lower = better)
- **Y-axis:** Constraint types supported (higher = more complete solution)

PRIME supports 12/12 constraint types (provider exclusions, feature system, subscription
constraints, renewable/non-renewable resources, distance, symbolic pricing).
Baselines support only 5/12 (resource demand, budget, max nodes, device type, provider name).

In [ ]:
# Pareto: time vs constraint coverage
pareto = feasible_all.groupby('technique').agg(
    median_time=('time_s', 'median'),
    constraints=('constraints', 'first'),
    family=('family', 'first'),
)

fig, ax = plt.subplots(figsize=(10, 7))
for family, color in FAMILY_COLORS.items():
    subset = pareto[pareto['family'] == family]
    ax.scatter(subset['median_time'], subset['constraints'],
              c=color, s=150, alpha=0.85, edgecolors='black', linewidth=0.5,
              label=family, zorder=5)
    for tech, row in subset.iterrows():
        label = tech.replace(f'{family}_', '').replace(family, '')
        offset = (8, 5) if family == 'PRIME' else (8, -8)
        ax.annotate(label, (row['median_time'], row['constraints']),
                    fontsize=8, ha='left', va='bottom', xytext=offset,
                    textcoords='offset points', fontweight='bold')

# Shade regions for hard constraints
ax.axhspan(5.5, 6.5, alpha=0.08, color='#e67e22', label='_nolegend_')  # RAOM4CC region
ax.axhspan(11.5, 12.5, alpha=0.08, color='#3498db', label='_nolegend_')  # PRIME region
ax.text(0.0002, 6.0, 'RAOM4CC: 6 hard +\n2 diagnostic = 8', fontsize=7, color='#e74c3c',
        ha='center', va='center', fontweight='bold')
ax.text(0.0002, 4.5, 'EdgeWiseCR: 6 hard\nconstraints', fontsize=7, color='#2ecc71',
        ha='center', va='center', fontweight='bold')
ax.text(1.0, 12.0, 'PRIME: 12 hard\nconstraints', fontsize=8, color='#3498db',
        ha='center', va='center', fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('Median Solver Time (seconds, log scale)')
ax.set_ylabel('Constraint Types (hard + diagnostic)')
ax.set_title('Pareto Front: Execution Time vs Constraint Coverage\n(lower-right = complete but slower; upper-left = fast but incomplete)')
ax.set_ylim(3, 13)
ax.legend(title='Family', loc='upper left')
plt.tight_layout()
plt.show()

## 7. Solution Completeness Analysis

Since baselines do not model provider exclusions, feature requirements, and other constraints,
their solutions may violate these constraints. A solution that ignores provider incompatibility
is not deployable in practice, even if it satisfies basic resource demands.

In [ ]:
# Constraint coverage visualization (nuanced: hard constraints + diagnostic metrics)
# Format: (name, PRIME, RAOM4CC, EdgeWiseCR) where values are:
#   'hard' = enforced as hard constraint
#   'diag' = computed as diagnostic metric (not enforced)
#   'no'   = not modeled
constraints = [
    ('Resource demand', 'hard', 'hard', 'hard'),
    ('Budget limit', 'hard', 'hard', 'hard'),
    ('Max nodes', 'hard', 'hard', 'hard'),
    ('Device type filter', 'hard', 'hard', 'hard'),
    ('Provider name', 'hard', 'hard', 'hard'),
    ('Capacity linking (a<=c*b)', 'hard', 'hard', 'hard'),
    ('Delay estimation', 'no', 'diag', 'no'),
    ('Energy estimation', 'no', 'diag', 'no'),
    ('Provider exclusions', 'hard', 'no', 'no'),
    ('Provider inclusion groups', 'hard', 'no', 'no'),
    ('Feature type system', 'hard', 'no', 'no'),
    ('Subscription min/max', 'hard', 'no', 'no'),
    ('Renewable/non-renewable', 'hard', 'no', 'no'),
    ('Distance constraint', 'hard', 'no', 'no'),
    ('Symbolic price expressions', 'hard', 'no', 'no'),
]

color_map = {'hard': '#2ecc71', 'diag': '#f39c12', 'no': '#e74c3c'}
label_map = {'hard': 'Yes', 'diag': 'Computed', 'no': 'No'}

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, p, r, e) in enumerate(constraints):
    for j, val in enumerate([p, r, e]):
        color = color_map[val]
        ax.scatter(j, i, c=color, s=200, marker='s', edgecolors='black', linewidth=0.5)
        ax.text(j, i, label_map[val], ha='center', va='center', fontsize=6,
               fontweight='bold', color='white')
ax.set_yticks(range(len(constraints))); ax.set_yticklabels([c[0] for c in constraints], fontsize=9)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['PRIME', 'RAOM4CC', 'EdgeWiseCR'], fontsize=10, fontweight='bold')
ax.set_title('Constraint Coverage: PRIME vs Baselines (hard constraints + diagnostic metrics)'); ax.invert_yaxis()
legend_elements = [mpatches.Patch(facecolor='#2ecc71', edgecolor='black', label='Hard constraint'),
                   mpatches.Patch(facecolor='#f39c12', edgecolor='black', label='Diagnostic (not enforced)'),
                   mpatches.Patch(facecolor='#e74c3c', edgecolor='black', label='Not modeled')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Solution Completeness Score
# PRIME: feasibility * constraint_coverage / 12
# Baselines: feasibility * 5 / 12 (they miss 7 constraint types)
feasible_all = all_data[all_data['feasible']].copy()
completeness = feasible_all.groupby('technique').agg(
    median_time=('time_s', 'median'),
    completeness=('constraints', 'first'),  # already 12 or 5
    family=('family', 'first'),
)
completeness['completeness_pct'] = completeness['completeness'] / 12 * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors = [tech_color(t) for t in completeness.index]
ax.barh(range(len(completeness)), completeness['completeness_pct'].values,
        color=colors, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(completeness)))
ax.set_yticklabels(completeness.index, fontsize=9)
ax.set_xlabel('Solution Completeness (%)')
ax.set_title('Solution Completeness: What Constraints Does Each Technique Enforce?')
ax.set_xlim(0, 110)
for i, v in enumerate(completeness['completeness_pct'].values):
    ax.text(v + 1, i, f'{v:.0f}%', va='center', fontsize=9, fontweight='bold')
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc='lower right', title='Family')
plt.tight_layout()
plt.show()

## 8. Key Findings

In [ ]:
print('=' * 80)
print('KEY FINDINGS')
print('=' * 80)

feas_rates = all_data.groupby('technique')['feasible'].mean() * 100
perfect = feas_rates[feas_rates == 100].index.tolist()
imperfect = feas_rates[feas_rates < 100].sort_values()
print(f'\n1. FEASIBILITY')
print(f'   Techniques with 100% feasibility: {len(perfect)}/{len(feas_rates)}')
for t, v in imperfect.items():
    print(f'   {t}: {v:.1f}%')

prime_med = feasible_all[feasible_all['technique'] == 'PRIME']['time_s'].median()
fastest_med = feasible_all.groupby('technique')['time_s'].median().drop('PRIME').min()
speedup = prime_med / fastest_med
print(f'\n2. EXECUTION TIME (solver-side)')
print(f'   PRIME median: {prime_med:.4f}s')
print(f'   Fastest baseline: {fastest_med:.6f}s ({speedup:.0f}x faster)')

print(f'\n3. CONSTRAINT COVERAGE (hard constraints)')
print(f'   PRIME: 12/12 hard constraints (100%)')
print(f'   RAOM4CC: 6/12 hard constraints (50%) + 2 diagnostic metrics (delay, energy)')
print(f'   EdgeWiseCR: 6/12 hard constraints (50%)')
print(f'   All three enforce: resource demand, budget, max nodes, device type,')
print(f'   provider name, capacity linking')
print(f'   Only PRIME enforces: provider exclusions, inclusion groups, feature system,')
print(f'   subscription min/max, renewable/non-renewable, distance, symbolic pricing')
print(f'   RAOM4CC additionally computes: delay estimation, energy estimation')
print(f'   (diagnostic metrics, not enforced as hard constraints)')

print(f'\n4. COST COMPARISON NOT VALID')
print(f'   Baselines do not enforce provider exclusions and other constraints.')
print(f'   Their cost figures reflect incomplete constraint satisfaction.')
print(f'   PRIME is the only technique guaranteeing fully deployable solutions.')